# 🟢 1.5 Multi-Surface Synchronization: Terminal Engine + VS Code / Cursor `/ide` Bridge
**Course:** Agentic CLI Engineering - Complete Claude Code Masterclass  
**Module:** Section 1 — Foundation & Memory Architecture  

---

## 🏛️ Architectural Objective
Using a terminal-native agent does **not** mean abandoning visual code review. By executing `/ide` inside Claude Code, you establish a real-time Inter-Process Communication (IPC) bridge between your **OS Terminal Engine (Brain & Hands)** and **VS Code / Cursor (Visual Display & LSP Sensor)**.

Once connected via `/ide`, three capabilities unlock automatically:
1. **Zero-Typing Selection Sync:** Highlighting lines in your editor automatically injects the file path, line range, and code snippet into Claude's context.
2. **Live LSP Diagnostic Streaming:** The IDE's Language Server (`Pyright`, `tsserver`, `Ruff`) streams live red-squiggle type errors directly into Claude Code.
3. **Native Graphical Diff Routing:** Proposed multi-file edits open as side-by-side diff tabs inside your IDE before writing to disk.

---

## Phase 1: Discovering & Binding the `/ide` Bridge
When you run `/ide` inside Claude Code, the CLI scans active local sockets and installed editor binaries (`code`, `cursor`, `windsurf`) to bind the current Git repository root to your open editor window.

Let's initialize our `lab_1_5_workspace`, check for installed IDE binaries on your machine, and generate the `.claude/ide_bridge_lock.json` handshake manifest.

In [ ]:
%%bash
# 0. Auto-sync required packages from root requirements.txt
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")
REQ_FILE=$( [ -f "requirements.txt" ] && echo "requirements.txt" || echo "../requirements.txt" )
if [ -f "$REQ_FILE" ]; then
    echo -e "\033[0;36m[SETUP] Syncing Python dependencies from $REQ_FILE...\033[0m"
    $PY_BIN -m pip install -q -r "$REQ_FILE"
fi

rm -rf lab_1_5_workspace
mkdir -p lab_1_5_workspace/fintech_monorepo/.claude
mkdir -p lab_1_5_workspace/fintech_monorepo/services/payments
cd lab_1_5_workspace/fintech_monorepo

echo -e "\033[1;32m=========================================================\033[0m"
echo -e "\033[1;32m [PHASE 1] '/ide' MULTI-SURFACE BRIDGE DISCOVERY         \033[0m"
echo -e "\033[1;32m=========================================================\033[0m"

DETECTED_IDE="vscode-compatible-headless"
for ide_bin in cursor code windsurf; do
    if command -v $ide_bin &>/dev/null; then
        echo -e "\033[0;32m[✔] IDE CLI Binary Detected :\033[0m $ide_bin ($(which$ide_bin))"
        DETECTED_IDE=$ide_bin
    else
        echo -e "\033[0;33m[i] IDE Binary ($ide_bin)   :\033[0m Not in PATH (Bridge simulation active)"
    fi
done

cat <<EOF > .claude/ide_bridge_lock.json
{
  "bridge_status": "CONNECTED",
  "ide_host": "$DETECTED_IDE",
  "workspace_root": "$(pwd)",
  "capabilities": {
    "selection_context_sync": true,
    "lsp_diagnostics_stream": true,
    "native_diff_viewer": true
  }
}
EOF

echo -e "\033[0;32m[✔] IPC Handshake Locked    :\033[0m .claude/ide_bridge_lock.json"
echo -e "\033[1;32m=========================================================\033[0m"

---
## Phase 2: Zero-Typing Editor Selection Sync
Without `/ide`, pointing the terminal agent to a specific function requires typing:
`Look at services/payments/payment_processor.py lines 11 to 18`.

With `/ide` active, highlighting lines `11–18` in VS Code or Cursor automatically attaches an `<ide_selection>` block to your prompt. Let's deploy `payment_processor.py` and run `ide_selection_sync.py` to see the exact payload injected into Claude Code when you highlight code with your mouse.

In [ ]:
%%writefile lab_1_5_workspace/fintech_monorepo/services/payments/payment_processor.py
from typing import Dict

def format_basis_points(bps: int) -> str:
    return f"{bps / 100:.2f}%"

def calculate_settlement_cents(amount_cents: int, fee_bps: int) -> Dict[str, int]:
    """
    Calculates integer cent settlement to prevent IEEE-754 floating-point drift.
    """
    # --- [START USER IDE HIGHLIGHT: LINE 11] ---
    # BUG: Float division introduces float return type instead of strict integer cents
    fee_cents = (amount_cents * fee_bps) / 10000.0
    net_cents = amount_cents - fee_cents
    return {
        "gross_cents": amount_cents,
        "fee_cents": fee_cents,
        "net_cents": net_cents
    }
    # --- [END USER IDE HIGHLIGHT: LINE 19] ---


In [ ]:
%%writefile lab_1_5_workspace/fintech_monorepo/ide_selection_sync.py
import sys
from pathlib import Path

def capture_ide_selection(file_path: Path, start_line: int, end_line: int, user_prompt: str) -> str:
    lines = file_path.read_text(encoding="utf-8").splitlines()
    selected_slice = "\n".join(
        f"{idx}: {lines[idx - 1]}" for idx in range(start_line, end_line + 1)
    )
    payload = f"""<ide_selection>
File: {file_path}
Lines: {start_line}-{end_line}
Selected Code:
{selected_slice}
</ide_selection>

User Prompt: {user_prompt}"""
    return payload

if __name__ == "__main__":
    target = Path("lab_1_5_workspace/fintech_monorepo/services/payments/payment_processor.py")
    prompt = "Fix this highlighted calculation so fee_cents and net_cents remain strict integers."
    print("\033[1;32m[IDE -> CLI BRIDGE] Auto-Injected Selection Payload (0 paths typed by user):\033[0m")
    print("-----------------------------------------------------------------------------")
    print(capture_ide_selection(target, 11, 18, prompt))


In [ ]:
%%bash
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")
$PY_BIN lab_1_5_workspace/fintech_monorepo/ide_selection_sync.py

---
## Phase 3: Live LSP Diagnostic Streaming (IDE "Red Squiggles" -> Terminal)
Your IDE runs background Language Servers (`Pyright` for Python, `tsserver` for TypeScript) that underline type errors in red.

In `payment_processor.py`, the function signature promises `-> Dict[str, int]`, but dividing by `10000.0` turns `fee_cents` and `net_cents` into `float` values! Let's run our **LSP Diagnostic Streamer** (`lsp_diagnostic_streamer.py`) to see how the `/ide` bridge feeds exact line-and-column type errors to Claude Code before a single test is even executed.

In [ ]:
%%writefile lab_1_5_workspace/fintech_monorepo/lsp_diagnostic_streamer.py
import ast
import json
import sys
from pathlib import Path

def inspect_python_lsp_diagnostics(file_path: Path) -> list[dict]:
    """
    Simulates Pyright / LSP static type analysis over the /ide bridge:
    Detects float division ('/' or float literals) inside functions annotated with Dict[str, int].
    """
    source = file_path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    diagnostics = []

    for node in ast.walk(tree):
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div):
            # True division (/) in Python always produces a float
            if node.lineno >= 10:
                diagnostics.append({
                    "file": str(file_path.name),
                    "line": node.lineno,
                    "col": node.col_offset + 1,
                    "severity": "ERROR",
                    "source": "Pyright-LSP (/ide bridge)",
                    "message": "Type 'float' cannot be assigned to return type 'Dict[str, int]'. Use integer floor division '//' instead of '/'."
                })
    return diagnostics

if __name__ == "__main__":
    target = Path(sys.argv[1])
    diags = inspect_python_lsp_diagnostics(target)
    if diags:
        print(f"\033[1;31m[LSP DIAGNOSTIC STREAM] {len(diags)} IDE Red-Squiggle Error(s) Detected in {target.name}:\033[0m")
        print(json.dumps(diags, indent=2))
        sys.exit(1)
    else:
        print(f"\033[1;32m[LSP DIAGNOSTIC STREAM] 0 IDE Errors in {target.name} (Clean Problems Panel!)\033[0m")
        sys.exit(0)


In [ ]:
%%bash
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")
$PY_BIN lab_1_5_workspace/fintech_monorepo/lsp_diagnostic_streamer.py \
    lab_1_5_workspace/fintech_monorepo/services/payments/payment_processor.py || true

---
## Phase 4: Visual Diff Rendering & Zero-Squiggle Verification
Armed with both the **Highlighted Selection** (Phase 2) and the **LSP Red-Squiggle Telemetry** (Phase 3), Claude Code generates a patch using integer floor division (`// 10000`) and pops open a side-by-side diff in your IDE.

Let's run `apply_ide_diff.py` to render the colorized unified diff, apply the patch, verify that the IDE's LSP `Problems` panel drops to **0 errors**, and execute `pytest`.

In [ ]:
%%writefile lab_1_5_workspace/fintech_monorepo/apply_ide_diff.py
import difflib
from pathlib import Path

TARGET = Path("lab_1_5_workspace/fintech_monorepo/services/payments/payment_processor.py")

PATCHED_CODE = '''from typing import Dict

def format_basis_points(bps: int) -> str:
    return f"{bps / 100:.2f}%"

def calculate_settlement_cents(amount_cents: int, fee_bps: int) -> Dict[str, int]:
    """
    Calculates integer cent settlement to prevent IEEE-754 floating-point drift.
    """
    # FIXED VIA /ide LSP BRIDGE: Strict integer floor division (//)
    fee_cents = (amount_cents * fee_bps) // 10000
    net_cents = amount_cents - fee_cents
    return {
        "gross_cents": amount_cents,
        "fee_cents": fee_cents,
        "net_cents": net_cents
    }
'''

old_lines = TARGET.read_text(encoding="utf-8").splitlines()
new_lines = PATCHED_CODE.splitlines()

print("\033[1;36m=== [IDE DIFF VIEWER TAB] payment_processor.py (Before vs. After) ===\033[0m")
for line in difflib.unified_diff(old_lines, new_lines, fromfile="a/payment_processor.py", tofile="b/payment_processor.py", lineterm=""):
    if line.startswith("+") and not line.startswith("+++"):
        print(f"\033[0;32m{line}\033[0m")
    elif line.startswith("-") and not line.startswith("---"):
        print(f"\033[0;31m{line}\033[0m")
    else:
        print(line)

# Accept Diff in IDE -> Write to disk
TARGET.write_text(PATCHED_CODE, encoding="utf-8")
print("\n\033[1;32m[✔] IDE Diff Accepted & Written to Disk.\033[0m")


In [ ]:
%%writefile lab_1_5_workspace/fintech_monorepo/services/payments/test_payment_processor.py
from payment_processor import calculate_settlement_cents

def test_integer_cent_precision():
    # $100.00 (10,000 cents) with 290 bps (2.90%) fee -> 290 cents fee, 9,710 cents net
    res = calculate_settlement_cents(amount_cents=10000, fee_bps=290)
    assert isinstance(res["fee_cents"], int)
    assert isinstance(res["net_cents"], int)
    assert res["fee_cents"] == 290
    assert res["net_cents"] == 9710


In [ ]:
%%bash
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")

# 1. Render IDE Diff & Apply Patch
$PY_BIN lab_1_5_workspace/fintech_monorepo/apply_ide_diff.py

echo ""
# 2. Re-Verify IDE LSP Diagnostics (Red Squiggles == 0)
$PY_BIN lab_1_5_workspace/fintech_monorepo/lsp_diagnostic_streamer.py \
    lab_1_5_workspace/fintech_monorepo/services/payments/payment_processor.py

echo ""
# 3. Run Deterministic Pytest Suite
echo -e "\033[1;32m[PYTEST] Verifying Strict Integer Settlement Guardrails...\033[0m"
$PY_BIN -m pytest lab_1_5_workspace/fintech_monorepo/services/payments/test_payment_processor.py -q --tb=short

---
## 🏆 Section 1 Complete: Foundation & Memory Architecture Locked!
Congratulations! Across Section 1 (Lectures 1.1 – 1.5), you have:
1. Replaced manual GUI copy-pasting with **Zero-Copy Bash Telemetry Loops** (`1.1`).
2. Initialized the global `@anthropic-ai/claude-code` engine, configured OpenRouter gateway manifests, and locked down `ripgrep` indexing with `/doctor` (`1.2`).
3. Engineered a `<150 Line` **`CLAUDE.md` Manifesto** enforcing Pydantic v2, SQLAlchemy 2.0 Async, and Tailwind (`1.3`).
4. Architected a **3-Tier Memory Hierarchy** (`Global`, `Repo-Root`, `Subdirectory`) with instant `#` memory injection (`1.4`).
5. Connected the terminal engine to VS Code / Cursor via **`/ide`** for live selection sync, LSP red-squiggle streaming, and visual diffs (`1.5`).

👉 **Up Next — Section 2: Interactive Session Mastery & Context Hygiene (`Lecture 2.1: Read-First Reconnaissance`)**!